<a href="https://colab.research.google.com/github/ricthomas65-cyber/QuantFinance/blob/main/EfficientFrontier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This python notebook calculates and plots an efficient frontier based on the inputs you give it. You only need to edit the tickers in Cell Block #2 and everything will update and run your output.

# Cell 1: Imports

In [ ]:
# ==========================================
# CELL 1: Imports
# ==========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf  #if you haven't installed yfinance before, you first need to run... !pip install yfinance
from scipy.optimize import minimize

# CELL 2: Download price data from Yahoo Finance

In [ ]:
# ==========================================
# CELL 2: Download price data from Yahoo Finance
# ==========================================
tickers = ['WMT', 'IBM', 'MSFT', 'GOOG', 'CI']

prices = yf.download(tickers, start='2018-01-01', end='2026-06-01',
                      interval='1mo', auto_adjust=True)['Close']

display(prices.head())
print(prices.shape)

# CELL 3: Calculate monthly returns, then annualize

In [ ]:
# ==========================================
# CELL 3: Calculate monthly returns, then annualize
# ==========================================
monthly_returns = prices.pct_change().dropna()

mean_returns = monthly_returns.mean() * 12          # annualized expected return per asset
cov_matrix = monthly_returns.cov() * 12              # annualized covariance matrix

# Reindex mean_returns and cov_matrix to match the 'tickers' list order
# This is crucial for correct alignment in portfolio calculations
mean_returns = mean_returns.reindex(tickers)
cov_matrix = cov_matrix.reindex(index=tickers, columns=tickers)

print("Annualized Expected Returns:")
print(mean_returns)
print("\nAnnualized Covariance Matrix:")
print(cov_matrix)

# CELL 4: Optimizer - minimize volatility for a given target return

In [ ]:
# ==========================================
# CELL 4: Optimizer - minimize volatility for a given target return
# ==========================================
def minimize_volatility(target_return, mean_returns, cov_matrix, n_assets):
    """Find the lowest-volatility portfolio that achieves exactly target_return."""

    def portfolio_vol(weights, cov_matrix):
        return np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))

    # Constraint 1: weights sum to 1 (fully invested)
    # Constraint 2: portfolio return equals the target return
    constraints = (
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        {'type': 'eq', 'fun': lambda w: np.dot(w, mean_returns) - target_return}
    )

    # Long-only: each weight between 0% and 100%
    bounds = tuple((0, 1) for _ in range(n_assets))

    # Start with an equal-weighted guess
    initial_guess = np.array([1 / n_assets] * n_assets)

    result = minimize(
        portfolio_vol,
        initial_guess,
        args=(cov_matrix,),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )
    return result

# CELL 5: Trace the efficient frontier

In [ ]:

# ==========================================
# CELL 5: Trace the efficient frontier
# ==========================================
n_assets = len(tickers)

min_ret = mean_returns.min()
max_ret = mean_returns.max()
target_returns = np.linspace(min_ret, max_ret, 30)  # 30 points along the frontier

frontier_volatility = []
frontier_weights = []

for target in target_returns:
    result = minimize_volatility(target, mean_returns, cov_matrix, n_assets)
    if result.success:
        frontier_volatility.append(result['fun'])
        frontier_weights.append(result['x'])
    else:
        frontier_volatility.append(np.nan)
        frontier_weights.append([np.nan] * n_assets)

# Build a clean results table: target return, volatility, and each asset's weight
frontier_df = pd.DataFrame(frontier_weights, columns=tickers)
frontier_df.insert(0, 'Target Return', target_returns)
frontier_df.insert(1, 'Volatility', frontier_volatility)

# Display weights as percentages for readability
display_df = frontier_df.copy()
display_df['Target Return'] = display_df['Target Return'].map('{:.2%}'.format)
display_df['Volatility'] = display_df['Volatility'].map('{:.2%}'.format)
for t in tickers:
    display_df[t] = display_df[t].map('{:.1%}'.format)

print(display_df)


# CELL 6: Chart the efficient frontier with individual asset points

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import yfinance as yf
# from scipy.optimize import minimize


# ==========================================
# CELL 6: Chart the efficient frontier with individual asset points
# ==========================================
asset_volatility = np.sqrt(np.diag(cov_matrix))  # each asset's own annualized volatility

fig, ax = plt.subplots(figsize=(10, 6))

# The frontier curve itself
ax.plot(frontier_df['Volatility'], frontier_df['Target Return'],
        color='navy', linewidth=2.5, label='Efficient Frontier')

# The individual assets as dots
ax.scatter(asset_volatility, mean_returns, color='crimson', s=90,
           zorder=5, label='Individual Assets')

# Label each dot with its ticker
for i, ticker in enumerate(mean_returns.index):
    ax.annotate(ticker, (asset_volatility[i], mean_returns.iloc[i]),
                textcoords="offset points", xytext=(8, 5), fontsize=10)

ax.set_xlabel('Volatility (Annualized Std Dev)')
ax.set_ylabel('Expected Return (Annualized)')
ax.set_title(f'Efficient Frontier: {', '.join(mean_returns.index.tolist())}')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.show()